# Chapter 40: trees

Fresh computations cover a split, boosting/HGB checks, forest variance, permutation/Shapley examples and seven models on the post hoc five-times-signal interaction panel. Shared tree/pipeline functions are used. Signal at $D_k$ closes, trade at $D_{k+1}$ closes, and the label ends/becomes available at $D_{k+6}$ in this constructed calendar. Training uses mature labels only. Two new figures are saved.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
import os, sys
from pathlib import Path
from uuid import uuid4
candidate = Path.cwd().resolve()
ROOT = next((p for p in (candidate, *candidate.parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start from the reader package root or a subdirectory")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("ch40_tree_models_" + uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
workspace_root = ROOT
print("Run directory:", RUN_DIR.relative_to(ROOT))
import importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def load_chapter(name):
    spec = importlib.util.spec_from_file_location(name, ROOT / "code" / f"{name}.py")
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

from quant_course import ml_pipeline as mp, tree_models as tm
ch40 = load_chapter('ch40_tree_models')

## 1. A single split

Check $\Delta=n_Ln_R(\bar y_L-\bar y_R)^2/n$. Adjacent-value midpoint thresholds give a best split at 4.5 with gain 18. The new plot uses squared percentage-return units.

In [ ]:
tab = pd.DataFrame(ch40.split_table(ch40.X8, ch40.Y8)).set_index("threshold")
display(tab.round(4))
assert np.allclose(tab["gain"], tab["gain_closed_form"]) and tab["gain"].idxmax() == 4.5 and tab["gain"].max() == 18.0
sk = ch40.sklearn_checks()
assert sk["stump_threshold"] == 4.5 and sk["stump_leaves"] == [-1.0, 2.0]
plt.bar(tab.index, tab["gain"], width=0.6, color="#247F82")
plt.xlabel("threshold s"); plt.ylabel("SSE reduction (%^2)"); plt.title("CONSTRUCTED 8-stock example"); plt.savefig(RUN_DIR / "ch40_tree_models_cell03.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 2. Boosting and regularized leaf values

Compute four hand boosting rounds at learning rate 0.5 and compare the second-round fitted vector with HGB. Inspect the separate L2 leaf-value check; regularization changes a leaf update, not a new market observation.

In [ ]:
b = ch40.boosting_by_hand(ch40.X8, ch40.Y8, 0.5, 4)
display(pd.DataFrame([{k: r[k] for k in ("round", "train_mse")} | {"threshold": r.get("threshold")} for r in b]).round(5))
assert np.allclose(b[2]["F"], sk["hgb_round2_pred"], atol=1e-6)        # sklearn HGB reproduces F_2
assert sk["hgb_l2_leaf_check"] == [-0.25, 1.25]                        # leaf -6/(4+4) = -0.75 around F0 = 0.5
print("F2 =", b[2]["F"])

## 3. Forest variance, permutation and Shapley

Check the correlation-dependent forest variance, a permutation toy example and two-feature Shapley allocations. Correlated trees retain a variance floor; a feature attribution is not automatically a causal effect.

In [ ]:
print({B: ch40.forest_variance(0.3, 1.0, B) for B in (1, 10, 100, float("inf"))})
assert np.isclose(ch40.forest_variance(0.3, 1.0, 10), 0.37) and np.isclose(ch40.forest_variance(0.1, 1.2, float("inf")), 0.12)
pt = ch40.permutation_toy()
print({k: round(v, 4) for k, v in pt.items() if k.startswith("imp")})
assert np.isclose(pt["imp_x1"], 1.56) and pt["imp_x2"] == 0.0
assert ch40.shapley_two(lambda a, b: 2 * a * b, (1.0, 1.0), (0.0, 0.0)) == (1.0, 1.0)
assert np.allclose(ch40.shapley_two(lambda a, b: 0.6 * a, (1.5, 1.5), (0.0, 0.0)), (0.9, 0.0))

## 4. Seven models on a post hoc 5× panel

The registered weaker-signal result kept every model's IC within ±0.01. This cell freshly runs the stronger interaction sensitivity, not the CLI's complete panel/importance grid. Historical timing guidance was about 3–6 minutes on one environment and is not a new timing guarantee. Models share folds, rows and ledger rules; zero/constant predictions stay in cash.

In [ ]:
close, labels, X = ch40.build_panel("interaction_strong")
folds = ch40.folds_for(labels)
out = ch40.run_models(close, labels, X, folds, importance=False)
rows = {m: {"R2_vs_mean_%": 100 * r["prediction"].get("oos_r2_vs_train_mean", np.nan), "IC": r["prediction"]["ic_mean"],
            "IC_t": r["prediction"]["ic_t"], "net10_SR": r["portfolio"]["0.001"]["sharpe"],
            "turnover": r["portfolio"]["0.001"]["turnover_ann"]} for m, r in out["metrics"].items()}
table = pd.DataFrame(rows).T
display(table.round(4))
assert round(table.loc["gbm", "IC"], 4) == 0.1190 and round(table.loc["ridge", "IC"], 4) == 0.0571
assert table.loc[["tree", "rf", "gbm"], "IC"].min() > table.loc[["ols", "ridge"], "IC"].max()
models = ["ols", "ridge", "tree", "rf", "gbm"]
plt.bar(models, table.loc[models, "IC"], color=["#7F7F7F", "#7F7F7F", "#247F82", "#247F82", "#247F82"])
plt.ylabel("mean daily Spearman IC (test)"); plt.title("CONSTRUCTED interaction panel, 5x signal"); plt.savefig(RUN_DIR / "ch40_tree_models_cell09.png", dpi=150, bbox_inches="tight"); plt.show(); plt.close()

## 5. Interpretation

The strong constructed signal produces higher tree IC and large absolute after-cost Sharpe values without realistic market meaning. Preserve the weaker registered result where trees did not find the interaction. The plotted comparison and split chart are new RUN_DIR assets.